# DIT5411: Traditional Chinese handwriting recognition
In Colab choose **Runtime → Change runtime type → GPU**, then run the cells in order. A full run has 13,065 outputs and 2,613,000 training draws per epoch. Checkpoints and reports persist on Drive. If Colab disconnects, rerun the notebook with the same settings; training resumes at the next saved epoch.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/DIT5411')
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['DIT5411_PROJECT_ROOT'] = str(ROOT)
os.environ['DIT5411_CACHE_ROOT'] = str(ROOT / 'data' / 'prepared_v2')
repo_url = 'https://github.com/HenryLok0/DIT5411_LokChunHang.git'
repo_ref = 'master'
source_zip = ROOT / 'DIT5411-assignment-source.zip'
REPO = Path('/content/DIT5411')
if not REPO.exists():
    if source_zip.is_file():
        shutil.unpack_archive(str(source_zip), '/content')
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', repo_ref, repo_url, str(REPO)], check=True)
assert (REPO / 'download_data.py').is_file(), 'Use the updated GitHub branch or place the source ZIP in MyDrive/DIT5411.'

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=REPO, check=True)

run('-m', 'pip', 'install', '-q', '-r', 'requirements.txt')
subprocess.run(['nvidia-smi'], check=True)
run('-c', 'import tensorflow as tf; assert tf.config.list_physical_devices("GPU"), "TensorFlow cannot see a GPU"')

## Prepare the full archive
The first run stores the verified ZIP and the packed image arrays on Drive. Raw images are not individually extracted. Allow several gigabytes of Drive space. Preprocessing and training read the files from local runtime storage for speed.

In [ ]:
local_cache = Path('/content/dit5411_prepared')
drive_cache = ROOT / 'data' / 'prepared_v2'
os.environ['DIT5411_CACHE_ROOT'] = str(local_cache)

def copy_cache(source, target):
    target.mkdir(parents=True, exist_ok=True)
    partial = target / 'transfer-partial'
    shutil.rmtree(partial, ignore_errors=True)
    shutil.copytree(source / 'full', partial)
    partial.rename(target / 'full')

if not (local_cache / 'full' / 'metadata.json').is_file():
    if (drive_cache / 'full' / 'metadata.json').is_file():
        copy_cache(drive_cache, local_cache)
    else:
        archive = ROOT / 'data' / 'all_data.zip'
        run('download_data.py', '--output', archive)
        local_zip = Path('/content/dit5411-full.zip')
        shutil.copyfile(archive, local_zip)
        run('prepare_data.py', '--archive', local_zip)

if not (drive_cache / 'full' / 'metadata.json').is_file():
    copy_cache(local_cache, drive_cache)

## Train A, B and C
Keep all settings identical across the three runs. If you want to change batch size or epoch budget after training has started, use a new project root or move the existing `runs/full` directory out of the way rather than overwriting it.

In [ ]:
EPOCHS = 5
BATCH = 128
MIXED = True
precision_flags = ['--mixed-precision'] if MIXED else []
for name in ('a', 'b', 'c'):
    run('train.py', '--model', name, '--epochs', EPOCHS, '--batch-size', BATCH, '--resume', *precision_flags)

## Compare final checkpoints on the holdout
Only run this after all three models finish. Copy the Markdown table into your GitHub submission as `reports/full-results.md` and link it from the README. The comparison and its chart are saved on Drive.

In [ ]:
from IPython.display import Image, Markdown, display
report = ROOT / 'reports' / 'full-results.md'
run('evaluate.py', '--batch-size', BATCH, '--report', report)
run('visualize_history.py')
display(Markdown(report.read_text(encoding='utf-8')))
display(Image(filename=str(ROOT / 'runs' / 'full' / 'model_comparison.png')))